# CNN-Transformer IDS: Complete Training & Evaluation Pipeline

This notebook provides a complete end-to-end pipeline for training and evaluating intrusion detection models:

## Part 1: Setup & Training
1. **Environment Setup** — Install dependencies, clone repo, configure paths
2. **Data Discovery** — Auto-detect and load CIC-IDS2017/2018 datasets
3. **Configuration** — Set hyperparameters and training options
4. **Train CNN Classifier** — Standalone Conv1D + Global Average Pooling
5. **Train CNN-Transformer** — Conv1D tokenizer + Transformer encoder

## Part 2: Evaluation Suite
6. **Confusion Matrix & Error Analysis** — Visualize predictions and analyze errors
7. **K-Fold Cross-Validation** — Temporal-chunk-aware CV for robustness testing
8. **Statistical Preprocessing Comparison** — Hypothesis testing for preprocessing impact
9. **Statistical Model Comparison** — Hypothesis testing between models

---
# Part 1: Setup & Training
---

In [ ]:
# Cell 1: Setup & install
import os, sys, glob, warnings, importlib
from importlib.metadata import version, PackageNotFoundError

warnings.filterwarnings('ignore')
os.environ['TORCHDYNAMO_DISABLE'] = '1'

REPO_URL = 'https://github.com/samaraweeramethun-eng/CNN-Transformer.git'
REPO_DIR = '/kaggle/working/cnn_transformer_only'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

!pip install -q shap joblib psutil
!pip install -q -e .

# Ensure this repo's package is imported (avoid collisions with similarly named installs)
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Clear stale modules from previous runs in this kernel
for name in list(sys.modules):
    if name == 'cnn_transformer_only' or name.startswith('cnn_transformer_only.'):
        del sys.modules[name]

import torch
cnn_transformer_only = importlib.import_module('cnn_transformer_only')

print('module file:', cnn_transformer_only.__file__)
try:
    pkg_ver = version('cnn-transformer-only')
except PackageNotFoundError:
    pkg_ver = getattr(cnn_transformer_only, '__version__', 'unknown')

print('\u2713 cnn_transformer_only version:', pkg_ver)
print('\u2713 PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('\u2713 GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2: Data discovery (auto-detect under /kaggle/input)
import pandas as pd

KAGGLE_DATASET_PATH = ''  # optional: set exact path to a CSV or folder

DATA_PATH = None
PEEK_CSV = None

if KAGGLE_DATASET_PATH and os.path.exists(KAGGLE_DATASET_PATH):
    if os.path.isdir(KAGGLE_DATASET_PATH):
        DATA_PATH = KAGGLE_DATASET_PATH
        folder_csvs = sorted(glob.glob(os.path.join(KAGGLE_DATASET_PATH, '*.csv')))
        if folder_csvs:
            folder_csvs.sort(key=os.path.getsize, reverse=True)
            PEEK_CSV = folder_csvs[0]
    else:
        DATA_PATH = KAGGLE_DATASET_PATH
        PEEK_CSV = KAGGLE_DATASET_PATH

if DATA_PATH is None:
    patterns = [
        '/kaggle/input/**/*TrafficForML*CICFlowMeter*.csv',
        '/kaggle/input/**/cicids*.csv',
        '/kaggle/input/**/*.csv',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    candidates = sorted(set(candidates))

    if candidates:
        candidates.sort(key=os.path.getsize, reverse=True)
        PEEK_CSV = candidates[0]
        candidate_dir = os.path.dirname(PEEK_CSV)
        same_dir_csvs = sorted(glob.glob(os.path.join(candidate_dir, '*.csv')))
        if len(same_dir_csvs) > 1:
            DATA_PATH = candidate_dir
            print('Auto-detected CSV folder:', DATA_PATH)
            print('CSV files in folder:', len(same_dir_csvs))
        else:
            DATA_PATH = PEEK_CSV
            print('Auto-detected CSV:', DATA_PATH)

# Fallback to repo sample if present
if DATA_PATH is None:
    if os.path.exists('data/cicids2017/cicids2017.csv'):
        DATA_PATH = 'data/cicids2017/cicids2017.csv'
    else:
        DATA_PATH = 'data/cicids2017/cicids2017_sample.csv'
    PEEK_CSV = DATA_PATH

print('Using input:', DATA_PATH)
if os.path.isfile(DATA_PATH):
    print('Size (MB):', os.path.getsize(DATA_PATH) / 1024**2)
else:
    print('Input is a folder; training will load all CSV files inside it.')

if PEEK_CSV is None:
    raise FileNotFoundError(f'No CSV file found for preview in: {DATA_PATH}')

df_peek = pd.read_csv(PEEK_CSV, nrows=5)
label_candidates = [c for c in df_peek.columns if 'label' in c.lower()]
print('Preview from:', PEEK_CSV)
print('Columns:', len(df_peek.columns))
label_col = label_candidates[0] if label_candidates else None
print('Label col:', label_col if label_col else 'NOT FOUND')

In [ ]:
# Cell 3: Configure models (shared by CNN classifier & CNN-Transformer)
from cnn_transformer_only.config import CNNTransformerConfig

USE_SAMPLE = ('sample' in DATA_PATH.lower())

# Memory guardrails for Kaggle: increase MAX_ROWS if you have headroom.
MAX_ROWS = 2_000_000 if not USE_SAMPLE else 300_000

cfg = CNNTransformerConfig(
    input_path=DATA_PATH,
    output_dir='/kaggle/working/artifacts',
    csv_chunksize=200_000,
    max_rows=MAX_ROWS,
    epochs=30 if not USE_SAMPLE else 5,
    batch_size=1024 if not USE_SAMPLE else 64,
    val_batch_size=2048 if not USE_SAMPLE else 128,
    lr=3e-5,
    weight_decay=5e-3,
    label_smoothing=0.1,
    dropout=0.3,
    num_workers=2,
    d_model=128 if not USE_SAMPLE else 64,
    conv_channels=64 if not USE_SAMPLE else 32,
    num_layers=2 if not USE_SAMPLE else 1,
    num_heads=4 if not USE_SAMPLE else 4,
    d_ff=512 if not USE_SAMPLE else 256,
    cnn_fc_dim=128 if not USE_SAMPLE else 64,
    ig_steps=32 if not USE_SAMPLE else 8,
    ig_samples=512 if not USE_SAMPLE else 128,
    undersampling_ratio=0.15,
    warmup_epochs=2,
    patience=4,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    split_strategy='temporal_chunks',
    num_blocks=50,
    chunk_size_blocks=5,
    purge_gap_blocks=0,
    checkpoint_metric='best_f1',
    attack_class_weight=0.0,
    grad_clip=0.5,
    grouped_split=True,
    correlation_threshold=0.95,
    skew_threshold=5.0,
    near_dup_decimals=3,
)

print('Mode:', 'SAMPLE' if USE_SAMPLE else 'FULL')
print('Output dir:', cfg.output_dir)
print('Data cap (rows):', cfg.max_rows if cfg.max_rows > 0 else 'ALL')
print('Config:', cfg.epochs, 'epochs | d_model', cfg.d_model,
      '| conv_channels', cfg.conv_channels, '| cnn_fc_dim', cfg.cnn_fc_dim,
      '| batch', cfg.batch_size)
print('Split: train=%.0f%% val=%.0f%% test=%.0f%% | strategy=%s'
      % (cfg.train_ratio*100, cfg.val_ratio*100, cfg.test_ratio*100, cfg.split_strategy))
print('  Temporal: %d blocks, chunk_size=%d, purge_gap=%d'
      % (cfg.num_blocks, cfg.chunk_size_blocks, cfg.purge_gap_blocks))
print('Preprocessing: grouped_split=%s | corr_thresh=%.2f | near_dup=%d'
      % (cfg.grouped_split, cfg.correlation_threshold, cfg.near_dup_decimals))

In [ ]:
# Cell 4: Train CNN Classifier
import gc, time, psutil
from cnn_transformer_only.training.cnn_only_trainer import train_cnn_classifier

os.makedirs(cfg.output_dir, exist_ok=True)

ram = psutil.virtual_memory()
print(f'System RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

gc.collect()
t0 = time.time()
cnn_only_path = train_cnn_classifier(cfg)
print('\nCheckpoint saved:', cnn_only_path)
print('Elapsed (min):', (time.time()-t0)/60)

# Display test metrics
cnn_only_ckpt = torch.load(cnn_only_path, map_location='cpu', weights_only=False)
cnn_only_metrics = cnn_only_ckpt.get('test_metrics', {})
print('\n=== CNN Classifier \u2014 Test Set Metrics ===')
for k in ['auc_roc','auc_pr','f1_score','recall','precision','accuracy']:
    val = cnn_only_metrics.get(k, None)
    print(f'{k}: {val:.4f}' if val is not None else f'{k}: None')
print('Best threshold:', cnn_only_ckpt.get('best_threshold', 0.5))

In [ ]:
# Cell 5: Train CNN-Transformer
from cnn_transformer_only.training.cnn_trainer import train_cnn_transformer

ram = psutil.virtual_memory()
print(f'System RAM: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'GPU VRAM: {(total-free)/1024**3:.1f} / {total/1024**3:.1f} GB')

gc.collect()
t0 = time.time()
cnn_transformer_path = train_cnn_transformer(cfg)
print('\nCheckpoint saved:', cnn_transformer_path)
print('Elapsed (min):', (time.time()-t0)/60)

# Display test metrics
ct_ckpt = torch.load(cnn_transformer_path, map_location='cpu', weights_only=False)
ct_metrics = ct_ckpt.get('test_metrics', {})
print('\n=== CNN-Transformer \u2014 Test Set Metrics ===')
for k in ['auc_roc','auc_pr','f1_score','recall','precision','accuracy']:
    val = ct_metrics.get(k, None)
    print(f'{k}: {val:.4f}' if val is not None else f'{k}: None')
print('Best threshold:', ct_ckpt.get('best_threshold', 0.5))

---
# Part 2: Evaluation Suite
---

In [ ]:
# Cell 6: Confusion Matrix Plots & Error Analysis
# ── Free memory from training cells before reloading ──────────────
import gc, psutil
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# Force free any leftover training tensors
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ram = psutil.virtual_memory()
print(f'RAM before load: {ram.used/1024**3:.1f} / {ram.total/1024**3:.1f} GB')

from cnn_transformer_only.evaluation.confusion_matrix import (
    plot_confusion_matrix,
    error_analysis_report,
)
from cnn_transformer_only.data import (
    binary_predictions_from_proba,
    load_cicids_feature_matrix,
    prepare_training_data,
)

# ── Load data & preprocess ────────────────────────────────────────
X, y, feature_cols, _, source_groups = load_cicids_feature_matrix(
    cfg.input_path,
    max_rows=cfg.max_rows,
    chunksize=cfg.csv_chunksize,
    return_source_groups=True,
)

(X_train, X_val, X_test, y_train, y_val, y_test,
 scaler, medians, feature_cols, prep_meta, test_block_map) = prepare_training_data(
    X, y, feature_cols, cfg, source_groups=source_groups,
)
# Free everything except test arrays
del X, y, source_groups, X_train, X_val, y_train, y_val, scaler, medians, prep_meta
gc.collect()

# ── Generate test predictions from CNN-Transformer checkpoint ─────
ckpt_path = cnn_transformer_path
from cnn_transformer_only.models.cnn_transformer import CNNTransformerIDS
model = CNNTransformerIDS(
    input_dim=X_test.shape[1],
    conv_channels=cfg.conv_channels,
    num_layers=cfg.num_layers,
    num_heads=cfg.num_heads,
    d_model=cfg.d_model,
    d_ff=cfg.d_ff,
    dropout=cfg.dropout,
)
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

test_ds = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False)

all_probs = []
with torch.no_grad():
    for data, _ in test_loader:
        logits = model(data)
        probs = F.softmax(logits, dim=1)[:, 1]
        all_probs.append(probs.numpy())
test_probs = np.concatenate(all_probs)
threshold = ckpt.get('best_threshold', 0.5)
test_preds = binary_predictions_from_proba(test_probs, threshold)

# Free model immediately
del model, ckpt, test_ds, test_loader, all_probs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Plot confusion matrix ─────────────────────────────────────────
plot_confusion_matrix(
    y_test, test_preds,
    title='CNN-Transformer — Test Set Confusion Matrix',
    output_dir=cfg.output_dir,
    filename='cnn_transformer_confusion_matrix.png',
)
import matplotlib.pyplot as plt
plt.show()

# ── Error analysis ────────────────────────────────────────────────
report = error_analysis_report(
    y_test, test_preds, test_probs,
    block_map=test_block_map,
    output_dir=cfg.output_dir,
    prefix='cnn_transformer',
)
print(f"High-confidence FP (prob>0.9): {report['high_conf_errors']['fp_above_0.9']}")
print(f"High-confidence FN (prob<0.1): {report['high_conf_errors']['fn_below_0.1']}")

# ── Free ALL Cell 6 data before CV cells ──────────────────────────
del X_test, y_test, test_block_map, test_probs, test_preds, report
gc.collect()
print(f'\nRAM after cleanup: {psutil.virtual_memory().used/1024**3:.1f} GB')

In [ ]:
# Cell 7: K-Fold Cross-Validation Test Harness
# Runs grouped k-fold CV using temporal chunks to prevent leakage.
# Tests both CNN Classifier and CNN-Transformer for comparison.
import gc, psutil
import numpy as np
import torch
from cnn_transformer_only.config import CNNTransformerConfig
from cnn_transformer_only.data import load_cicids_feature_matrix
from cnn_transformer_only.evaluation.cross_validation import grouped_kfold_cv

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'RAM before CV: {psutil.virtual_memory().used/1024**3:.1f} GB')

# ── Cap rows for CV to avoid OOM (500K is usually sufficient) ─────
CV_MAX_ROWS = min(cfg.max_rows, 500_000) if cfg.max_rows > 0 else 500_000

config_cv = CNNTransformerConfig(
    input_path=cfg.input_path,
    output_dir=cfg.output_dir,
    split_strategy=cfg.split_strategy,
    num_blocks=cfg.num_blocks,
    chunk_size_blocks=cfg.chunk_size_blocks,
    near_dup_decimals=cfg.near_dup_decimals,
    random_state=cfg.random_state,
    max_rows=CV_MAX_ROWS,
    epochs=10,        # fewer epochs per fold for speed + memory
    batch_size=512,   # smaller batch to reduce peak GPU memory
    val_batch_size=1024,
)

X_cv, y_cv, feat_cv, _, sg_cv = load_cicids_feature_matrix(
    config_cv.input_path, max_rows=config_cv.max_rows,
    chunksize=config_cv.csv_chunksize, return_source_groups=True,
)

# Create sequential blocks
num_blocks = config_cv.num_blocks
block_groups = (np.arange(len(y_cv)) * num_blocks // len(y_cv)).astype(np.int32)

# Remove exact + near duplicates before CV
row_view = np.ascontiguousarray(X_cv).view(
    np.dtype((np.void, X_cv.dtype.itemsize * X_cv.shape[1]))
).ravel()
_, uniq_idx = np.unique(row_view, return_index=True)
uniq_idx.sort()
X_clean = X_cv[uniq_idx]
y_clean = y_cv[uniq_idx]
block_groups_clean = block_groups[uniq_idx]
del X_cv, y_cv, row_view, uniq_idx; gc.collect()

nd_dec = config_cv.near_dup_decimals
if nd_dec > 0:
    X_rounded = np.round(np.nan_to_num(X_clean, nan=-999.0), decimals=nd_dec)
    rounded_view = np.ascontiguousarray(X_rounded).view(
        np.dtype((np.void, X_rounded.dtype.itemsize * X_rounded.shape[1]))
    ).ravel()
    _, nd_idx = np.unique(rounded_view, return_index=True)
    nd_idx.sort()
    X_clean = X_clean[nd_idx]
    y_clean = y_clean[nd_idx]
    block_groups_clean = block_groups_clean[nd_idx]
    del X_rounded, rounded_view, nd_idx; gc.collect()

X_clean[np.isinf(X_clean)] = np.nan

print(f'CV data after cleaning: {len(y_clean):,} samples, {X_clean.shape[1]} features')
print(f'Unique blocks: {len(np.unique(block_groups_clean))}')
print(f'RAM: {psutil.virtual_memory().used/1024**3:.1f} GB')

# ── Run CV for CNN-Transformer ────────────────────────────────────
print('\n' + '=' * 70)
print('  K-FOLD CV: CNN-Transformer')
print('=' * 70)
cv_cnn_transformer = grouped_kfold_cv(
    X_clean, y_clean, block_groups_clean, config_cv,
    n_folds=5, model_type='cnn_transformer',
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Run CV for CNN Classifier ─────────────────────────────────────
print('\n' + '=' * 70)
print('  K-FOLD CV: CNN Classifier')
print('=' * 70)
cv_cnn_classifier = grouped_kfold_cv(
    X_clean, y_clean, block_groups_clean, config_cv,
    n_folds=5, model_type='cnn_classifier',
)

# Keep X_clean, y_clean, block_groups_clean alive for walk-forward (Cell 8)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'\nRAM after CV: {psutil.virtual_memory().used/1024**3:.1f} GB')

In [ ]:
# Cell 8: Walk-Forward (Rolling Window) Validation
# Tests model stability over sequential time blocks to detect concept drift.
# Uses a sliding window: train on blocks [i..i+30), test on [i+30..i+35),
# stepping forward by 5 blocks each iteration.
import gc, psutil
import torch
from cnn_transformer_only.evaluation.cross_validation import walk_forward_cv

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'RAM before walk-forward: {psutil.virtual_memory().used/1024**3:.1f} GB')

# X_clean, y_clean, block_groups_clean are from Cell 7
wf_results = walk_forward_cv(
    X_clean, y_clean, block_groups_clean, config_cv,
    train_window_blocks=30,
    test_window_blocks=5,
    step_blocks=5,
    model_type='cnn_transformer',
    plot=True,
)

print(f'\nWalk-forward steps completed: {wf_results["n_steps"]}')
print(f'Mean F1:  {wf_results["mean_metrics"]["f1_score"]["mean"]:.4f}')
print(f'Mean AUC: {wf_results["mean_metrics"]["auc_roc"]["mean"]:.4f}')

drift = wf_results['degradation']
if drift['drifting']:
    print(f'\n*** CONCEPT DRIFT DETECTED ***')
    print(f'  F1 slope per step: {drift["f1_slope_per_step"]:.4f}')
    print(f'  Total F1 change:   {drift["f1_total_change"]:.4f}')
else:
    print(f'\nNo significant concept drift detected.')
    print(f'  F1 slope per step: {drift["f1_slope_per_step"]:.4f}')

# Free cleaned data — no longer needed by subsequent cells
del X_clean, y_clean, block_groups_clean
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'\nRAM after walk-forward: {psutil.virtual_memory().used/1024**3:.1f} GB')

In [ ]:
# Cell 9: Statistical Test — Preprocessing Comparison (with/without dedup)
# Compares model performance BEFORE and AFTER near-duplicate removal
# using paired t-test and Wilcoxon signed-rank test across CV folds.
import gc, psutil
import numpy as np
import torch
from cnn_transformer_only.config import CNNTransformerConfig
from cnn_transformer_only.data import load_cicids_feature_matrix
from cnn_transformer_only.evaluation.cross_validation import grouped_kfold_cv
from cnn_transformer_only.evaluation.statistical_tests import compare_preprocessing

# X_clean already freed by Cell 8 (walk-forward)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f'RAM before Cell 9: {psutil.virtual_memory().used/1024**3:.1f} GB')

# Condition A: WITH near-duplicate removal (already computed in Cell 7)
cv_with_dedup = cv_cnn_transformer

# Condition B: WITHOUT near-duplicate removal
# Use same capped row limit as Cell 7 for consistency + memory safety
CV_MAX_ROWS = min(cfg.max_rows, 500_000) if cfg.max_rows > 0 else 500_000

config_no_dedup = CNNTransformerConfig(
    input_path=cfg.input_path,
    output_dir=cfg.output_dir,
    split_strategy=cfg.split_strategy,
    num_blocks=cfg.num_blocks,
    chunk_size_blocks=cfg.chunk_size_blocks,
    random_state=cfg.random_state,
    max_rows=CV_MAX_ROWS,
    epochs=10,
    batch_size=512,
    val_batch_size=1024,
    near_dup_decimals=0,  # DISABLE near-duplicate removal
)

X_raw, y_raw, feat_raw, _, sg_raw = load_cicids_feature_matrix(
    config_no_dedup.input_path, max_rows=config_no_dedup.max_rows,
    chunksize=config_no_dedup.csv_chunksize, return_source_groups=True,
)

num_blocks = config_no_dedup.num_blocks
block_groups_raw = (np.arange(len(y_raw)) * num_blocks // len(y_raw)).astype(np.int32)

# Remove only exact duplicates (not near-dups)
row_view = np.ascontiguousarray(X_raw).view(
    np.dtype((np.void, X_raw.dtype.itemsize * X_raw.shape[1]))
).ravel()
_, uniq_idx = np.unique(row_view, return_index=True)
uniq_idx.sort()
X_no_dedup = X_raw[uniq_idx]
y_no_dedup = y_raw[uniq_idx]
bg_no_dedup = block_groups_raw[uniq_idx]
X_no_dedup[np.isinf(X_no_dedup)] = np.nan
del X_raw, y_raw, row_view, uniq_idx; gc.collect()

print(f'Data WITHOUT near-dedup: {len(y_no_dedup):,} samples')

cv_without_dedup = grouped_kfold_cv(
    X_no_dedup, y_no_dedup, bg_no_dedup, config_no_dedup,
    n_folds=5, model_type='cnn_transformer',
)
del X_no_dedup, y_no_dedup, bg_no_dedup; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Statistical comparison
preprocess_results = compare_preprocessing(
    cv_without_dedup,
    cv_with_dedup,
    label_before='Without Near-Dedup',
    label_after='With Near-Dedup',
    metrics=['f1_score', 'auc_roc', 'auc_pr', 'precision', 'recall'],
)
print(f'\nRAM after Cell 9: {psutil.virtual_memory().used/1024**3:.1f} GB')

In [ ]:
# Cell 10: Statistical Test \u2014 Model Comparison (CNN Classifier vs CNN-Transformer)
# Compares the two best-performing models using paired t-test and
# Wilcoxon signed-rank test on the SAME k-fold CV splits.
# Both models were trained on identical fold assignments in Cell 7.
from cnn_transformer_only.evaluation.statistical_tests import compare_models_statistical

# cv_cnn_classifier and cv_cnn_transformer are from Cell 7
model_comparison_results = compare_models_statistical(
    cv_cnn_classifier,
    cv_cnn_transformer,
    label_a='CNN Classifier',
    label_b='CNN-Transformer',
    metrics=['f1_score', 'auc_roc', 'auc_pr', 'precision', 'recall'],
)

# Summary
print('\nKey takeaways:')
for r in model_comparison_results:
    m = r['metric']
    diff = r['mean_diff']
    p_t = r['paired_t']['p_value']
    sig = r['paired_t']['significant']
    direction = 'CNN Classifier' if diff > 0 else 'CNN-Transformer'
    if sig:
        print(f'  {m}: {direction} is significantly better (p={p_t:.4f}, diff={abs(diff):.4f})')
    else:
        print(f'  {m}: No significant difference (p={p_t:.4f})')